# 02. Trend

Every Prophet forecast starts from a trend: the long-run direction of the series once seasonality and events are taken out. This notebook shows how the trend is built, how to make it stiffer or more flexible, how to cap its growth and how to stop a one-off shock from bending the forecast.

- What the trend is
- Automatic changepoints and their flexibility
- Where changepoints may sit: `changepoint_range`
- Specifying changepoints by hand
- Saturating growth with a cap and a floor
- Flat growth
- Handling shocks: Melbourne pedestrians and COVID-19

Previous: [01. Your data](01_your_data.ipynb) | Next: [03. Seasonality and holidays](03_seasonality_and_holidays.ipynb)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from prophet import Prophet, forecast
from prophet.plot import add_changepoints_to_plot, plot_forecast_component
from prophet.quick import ForecastInputError

from helpers import load_example, quiet_logs

quiet_logs()

## 1. What the trend is

By default the trend is a piecewise linear curve. Prophet scatters candidate changepoints, dates where the slope is allowed to change, across the history and gives the slope changes a sparse prior, so most candidates stay unused. Below is the default fit on Peyton Manning's Wikipedia page views (log scale), forecasting a year ahead. We pass `freq="D"` because the series has some missing days and `forecast()` would otherwise warn about irregular spacing.

In [ ]:
df = load_example("peyton_manning")
df.tail()

In [ ]:
result = forecast(df, horizon=365, freq="D")
m, fcst = result.model, result.predictions
fcst[["ds", "trend", "yhat"]].tail()

In [ ]:
_ = plot_forecast_component(m, fcst, "trend")

The candidates are in `m.changepoints` and the fitted slope changes are in `m.params["delta"]`. The helper below lists the changepoints that were actually used, with each slope change converted to units of `y` per year so that it is readable.

In [ ]:
def changepoint_table(m: Prophet, threshold: float = 0.01) -> pd.DataFrame:
    """Changepoints that bend the trend, largest first.

    ``delta`` is the model's own scaled slope change; ``threshold`` is on that
    scale and matches the markers drawn by ``add_changepoints_to_plot``.
    ``slope_change_per_year`` restates it in units of y per year.
    """
    delta = m.params["delta"].mean(axis=0)
    years = m.t_scale / pd.Timedelta(days=365.25)
    table = pd.DataFrame(
        {
            "ds": np.sort(m.changepoints.to_numpy()),
            "delta": delta,
            "slope_change_per_year": delta * m.y_scale / years,
        }
    )
    used = table[table["delta"].abs() > threshold]
    return used.sort_values("delta", key=abs, ascending=False, ignore_index=True)


table = changepoint_table(m)
print(f"{len(m.changepoints)} candidate changepoints, {len(table)} used")
table.head()

Takeaway: the trend rises into early 2012, when the series peaks, and then drifts down. The forecast simply extends the last segment.

## 2. Automatic changepoints and their flexibility

`changepoint_prior_scale` (default 0.05) is the main stiffness control. A small value keeps the trend nearly straight; a large one lets it bend at many candidates. The same data is fitted below with 0.001, the default and 0.5. Dashed lines mark the changepoints that were used.

In [ ]:
scales = [0.001, 0.05, 0.5]
by_scale = {s: forecast(df, 365, freq="D", changepoint_prior_scale=s) for s in scales}

fig, axes = plt.subplots(len(scales), 1, figsize=(10, 12), sharex=True, layout="constrained")
for ax, (scale, r) in zip(axes, by_scale.items()):
    r.plot(ax=ax, title=f"changepoint_prior_scale = {scale}", subtitle=False, annotate=False)
    add_changepoints_to_plot(ax, r.model, r.predictions)

In [ ]:
def fit_summary(r) -> dict[str, float]:
    """In-sample error, changepoints used and the interval width at the end of the horizon."""
    fitted = r.predictions.merge(r.history, on="ds")
    last = r.future.iloc[-1]
    return {
        "changepoints_used": len(changepoint_table(r.model)),
        "in_sample_rmse": float(np.sqrt(((fitted["yhat"] - fitted["y"]) ** 2).mean())),
        "interval_width_at_horizon": float(last["yhat_upper"] - last["yhat_lower"]),
    }


pd.DataFrame.from_dict({s: fit_summary(r) for s, r in by_scale.items()}, orient="index").rename_axis("changepoint_prior_scale").round(3)

Takeaway: more flexibility fits the past better but makes the future less certain, because the model now allows the trend to bend again. In-sample error cannot choose the value for you; notebook 06 does that with cross-validation.

## 3. Where changepoints may sit: `changepoint_range`

Candidates are placed evenly over the first 80% of the history by default (`changepoint_range=0.8`, with `n_changepoints=25` of them). The last fifth is kept clear so the model cannot read the final few weeks of noise as a new trend. Raising the range lets the trend react to recent data, at that risk.

In [ ]:
ranges = [0.8, 0.95]
by_range = {cr: forecast(df, 365, freq="D", changepoint_range=cr) for cr in ranges}

fig, axes = plt.subplots(len(ranges), 1, figsize=(10, 8), sharex=True, layout="constrained")
for ax, (cr, r) in zip(axes, by_range.items()):
    r.plot(ax=ax, title=f"changepoint_range = {cr}", subtitle=False, annotate=False)
    add_changepoints_to_plot(ax, r.model, r.predictions)

In [ ]:
pd.DataFrame.from_dict(
    {
        cr: {
            "last_candidate": r.model.changepoints.max().date(),
            "last_changepoint_used": changepoint_table(r.model)["ds"].max().date(),
            "trend_at_horizon": round(float(r.predictions["trend"].iloc[-1]), 2),
        }
        for cr, r in by_range.items()
    },
    orient="index",
).rename_axis("changepoint_range")

Takeaway: with 0.8 the trend is a straight line after the last changepoint used (December 2013); with 0.95 it can still bend in 2015. Here the forecast barely moves (7.19 against 7.24 at the horizon) because the recent data holds no strong new trend; the setting matters more when a series turns near its end.

## 4. Specifying changepoints by hand

If you know when something changed, pass `changepoints=` with those dates. They replace the automatic grid, so `n_changepoints` and `changepoint_range` are ignored, and every date must fall inside the training data. Here the two dates are the start of the 2011 NFL season, which Manning missed through injury, and his signing with the Denver Broncos in March 2012. They are illustrative choices, not fitted ones.

In [ ]:
manual = forecast(df, 365, freq="D", changepoints=["2011-09-08", "2012-03-20"])

fig = manual.plot(annotate=False, title="Two hand-picked changepoints", subtitle=False)
add_changepoints_to_plot(fig.gca(), manual.model, manual.predictions)
changepoint_table(manual.model)

Takeaway: the trend can now bend only on those two dates. That is a strong statement, so use it for known events rather than to force a shape.

## 5. Saturating growth with a cap and a floor

Logistic growth flattens out at a carrying capacity (`cap`) and optionally a lower limit (`floor`). Both are columns that must exist for every row of the history and the future frame. `forecast()` has no way to take them and rejects the request, which is why this section uses `Prophet` directly.

In [ ]:
w = load_example("wp_log_r")
try:
    forecast(w, 365, freq="D", growth="logistic")
except ForecastInputError as err:
    print(err)

The data is log page views of the Wikipedia page for R. It is not literally capacity-bound, so read the cap here as an assumption we choose to demonstrate the mechanics. `fit_logistic` takes `cap` and `floor` as functions of the dates, so they can be constant or moving, and returns the model with its three-year forecast.

In [ ]:
def constant(value: float):
    return lambda ds: pd.Series(value, index=ds.index)


def rising(start: float, per_year: float, since: str):
    origin = pd.Timestamp(since)
    return lambda ds: start + per_year * (ds - origin).dt.days.clip(lower=0) / 365.25


def fit_logistic(df: pd.DataFrame, cap, floor, periods: int = 1095):
    train = df.assign(cap=cap(df["ds"]), floor=floor(df["ds"]))
    m = Prophet(growth="logistic").fit(train)
    future = m.make_future_dataframe(periods=periods)
    future = future.assign(cap=cap(future["ds"]), floor=floor(future["ds"]))
    return m, m.predict(future)


scenarios = {
    "cap 8.3": constant(8.3),
    "cap 12": constant(12.0),
    "cap 9 rising 0.5 a year from 2016": rising(9.0, 0.5, "2016-01-01"),
}
logistic = {name: fit_logistic(w, cap, constant(3.0)) for name, cap in scenarios.items()}

fig, axes = plt.subplots(len(logistic), 1, figsize=(10, 12), sharex=True, layout="constrained")
for ax, (name, (m_log, f_log)) in zip(axes, logistic.items()):
    m_log.plot(f_log, ax=ax, title=f"{name}, floor 3", subtitle=False, annotate=False)

In [ ]:
linear = forecast(w, 1095, freq="D")
ends = {name: f["trend"].iloc[-1] for name, (_, f) in logistic.items()}
ends["linear growth, no cap"] = linear.predictions["trend"].iloc[-1]
pd.Series(ends, name="trend at the end of 2018").round(2).to_frame()

Takeaway: the series has already levelled off, so the two constant caps give almost the same forecast. The rising cap changes the answer, which shows the cap is an assumption you supply and not something the data pins down. Linear growth has no limit at all; it ends lower here only because its last segment slopes slightly down.

## 6. Flat growth

`growth="flat"` fits a constant level and no slope, so there are no changepoints to tune. It suits series that vary around a stable level, and `forecast()` accepts it. To see the contrast we use only the Peyton Manning data from 2013, where the direction is unclear, and compare it with the default linear trend.

In [ ]:
recent = df[df["ds"] >= "2013-01-01"]
trend_models = {
    "linear": forecast(recent, 365, freq="D"),
    "flat": forecast(recent, 365, freq="D", growth="flat"),
}

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True, sharey=True, layout="constrained")
for ax, (name, r) in zip(axes, trend_models.items()):
    r.plot(ax=ax, title=f"growth = {name!r}", subtitle=False, annotate=False)

In [ ]:
pd.DataFrame.from_dict(
    {
        name: {
            "trend_at_data_end": r.predictions["trend"].loc[r.predictions["ds"] == recent["ds"].max()].iloc[0],
            "trend_at_horizon": r.predictions["trend"].iloc[-1],
        }
        for name, r in trend_models.items()
    },
    orient="index",
).round(2)

Takeaway: the flat model forecasts from the average level of the three years (8.05) and so sits above the latest level (7.56); the linear model carries the recent decline forward. Neither is "right": pick flat only when you believe the level is stable.

## 7. Handling shocks: Melbourne pedestrians and COVID-19

A shock such as a lockdown looks like a trend change to Prophet, and the forecast may then extrapolate the crash or the rebound. Here are daily pedestrian counts from a Melbourne sensor.

In [ ]:
df_ped = load_example("pedestrians")
df_ped.tail()

In [ ]:
ax = df_ped.set_index("ds")["y"].plot(figsize=(10, 4), legend=False, xlabel="")

There are two shocks: the first lockdown from late March 2020 and, after a partial recovery, a series of shorter lockdowns into 2021. First the default model, with no adjustments.

In [ ]:
default = forecast(df_ped, horizon=365)

The second model treats each lockdown as a one-off holiday-style event, using `holidays=`, which `forecast()` passes straight to `Prophet`. A holiday frame needs `holiday` and `ds`, and optional `lower_window` and `upper_window` to extend an event over several days. Each lockdown gets a window that spans its length, so the dip is credited to the event rather than the trend. The dates follow the upstream Prophet documentation for the Victorian lockdowns and are approximate.

In [ ]:
bounds = pd.DataFrame(
    {
        "holiday": ["lockdown_1", "lockdown_2", "lockdown_3", "lockdown_4"],
        "ds": pd.to_datetime(["2020-03-21", "2020-07-09", "2021-02-13", "2021-05-28"]),
        "end": pd.to_datetime(["2020-06-06", "2020-10-27", "2021-02-17", "2021-06-10"]),
    }
)
lockdowns = bounds.assign(
    lower_window=0,
    upper_window=(bounds["end"] - bounds["ds"]).dt.days,
).drop(columns="end")
lockdowns

In [ ]:
as_events = forecast(df_ped, horizon=365, holidays=lockdowns)

The third model adds a changepoint at the end of each lockdown, so the trend may change slope after a shock instead of carrying the recovery slope forward. Because manual changepoints replace the automatic grid, we also supply ten evenly spaced candidates before 2020 to keep the pre-COVID trend flexible.

In [ ]:
before = pd.date_range(df_ped["ds"].min() + pd.Timedelta(days=30), "2020-02-01", periods=10)
after_shock = [*before, *bounds["end"]]
with_changepoints = forecast(df_ped, horizon=365, holidays=lockdowns, changepoints=after_shock)

In [ ]:
models = {
    "Default": default,
    "Lockdowns as events": as_events,
    "Events plus changepoints after each lockdown": with_changepoints,
}

fig, axes = plt.subplots(len(models), 1, figsize=(10, 12), sharex=True, layout="constrained")
for ax, (name, r) in zip(axes, models.items()):
    r.plot(ax=ax, title=name, subtitle=False, annotate=False)
    add_changepoints_to_plot(ax, r.model, r.predictions)

The trend on its own, with the uncertainty it adds to the forecast, and the numbers behind it.

In [ ]:
fig, axes = plt.subplots(len(models), 1, figsize=(10, 9), sharex=True, layout="constrained")
for ax, (name, r) in zip(axes, models.items()):
    _ = plot_forecast_component(r.model, r.predictions, "trend", ax=ax, title=name, annotate=False)

In [ ]:
pd.DataFrame.from_dict(
    {
        name: {
            "trend_at_data_end": r.predictions["trend"].loc[r.predictions["ds"] == df_ped["ds"].max()].iloc[0],
            "trend_at_horizon": r.predictions["trend"].iloc[-1],
            "mean_forecast": r.future["yhat"].mean(),
        }
        for name, r in models.items()
    },
    orient="index",
).round(0).astype(int)

Takeaway: the default model reads the rebound as a trend and keeps it going, to about 41,000 by mid-2022, above the roughly 36,000 level of 2017 to 2019. Treating the lockdowns as events gives about 31,000, and adding the changepoints about 32,500.

Limits: no holdout can settle which of these is right, because the shock is unique and the data ends in the middle of the recovery. The events also assume no further lockdowns; a new one would have to be added to the holiday frame, with its dates, before the forecast could include it. Treat the three forecasts as a range of scenarios.

## Next

The trend is only part of a forecast. Notebook [03. Seasonality and holidays](03_seasonality_and_holidays.ipynb) covers the repeating patterns and the special days on top of it.